# RAG Notebook 4 — Final Comparison

Run this **after** Notebooks 1, 2, and 3 have finished.

Loads the 3 saved `*_metrics.json` files and prints the full comparison table.

**No GPU needed — CPU only.**

## ── CELL 1 ── Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

Mounted at /content/drive
✅ Drive mounted


## ── CELL 2 ── Paths

In [2]:
import json, os
import pandas as pd
import numpy as np

RESULTS_DIR = '/content/drive/MyDrive/AgriQA/rag_comparison_results'

muril_path  = f'{RESULTS_DIR}/muril_rag_metrics.json'
xlmr_path   = f'{RESULTS_DIR}/xlmr_rag_metrics.json'
sarvam_path = f'{RESULTS_DIR}/sarvam_rag_metrics.json'

for p in [muril_path, xlmr_path, sarvam_path]:
    assert os.path.exists(p), f'Not found: {p} — run the corresponding notebook first'
print('✅ All 3 metric files found')

✅ All 3 metric files found


## ── CELL 3 ── Load and compare

In [3]:
with open(muril_path)  as f: muril  = json.load(f)
with open(xlmr_path)   as f: xlmr   = json.load(f)
with open(sarvam_path) as f: sarvam = json.load(f)

rows = []
for m in [muril, xlmr, sarvam]:
    rows.append({
        'Model':             m['model'].split('/')[-1],
        'Type':              m['type'],
        'BLEU':              m['bleu'],
        'BERTScore F1':      m['bertscore_f1'],
        'Answer Similarity': m['answer_similarity'],
        'Context Relevance': m['context_relevance'],
        'Faithfulness':      m['faithfulness'],
    })

df = pd.DataFrame(rows)

print()
print('=' * 80)
print('  FINAL RAG COMPARISON — 3 Models × 5 Metrics')
print('=' * 80)
print(df.to_string(index=False))
print('=' * 80)

for metric in ['BLEU','BERTScore F1','Answer Similarity','Context Relevance','Faithfulness']:
    best = df.loc[df[metric].idxmax(), 'Model']
    print(f'Best {metric:20s}: {best}')


  FINAL RAG COMPARISON — 3 Models × 5 Metrics
                  Model          Type   BLEU  BERTScore F1  Answer Similarity  Context Relevance  Faithfulness
       muril-base-cased extractive_qa 1.0007        0.6047             0.5410               0.83        0.5456
xlm-roberta-base-squad2 extractive_qa 4.8908        0.6826             0.6285               0.83        0.6040
         sarvam-2b-v0.5    generative 2.1260        0.6730             0.7548               0.83        0.7091
Best BLEU                : xlm-roberta-base-squad2
Best BERTScore F1        : xlm-roberta-base-squad2
Best Answer Similarity   : sarvam-2b-v0.5
Best Context Relevance   : muril-base-cased
Best Faithfulness        : sarvam-2b-v0.5


## ── CELL 4 ── Metric explanations

In [4]:
print("""
METRIC EXPLANATIONS
───────────────────
BLEU              : n-gram overlap between predicted and gold answer.
                    Higher = more word-for-word match.
                    Range: 0–100. Generative models score lower even when correct.

BERTScore F1      : Semantic similarity using multilingual BERT embeddings (lang=kn).
                    More reliable than BLEU for Kannada paraphrased answers.
                    Range: 0–1. > 0.80 is very good.

Answer Similarity : Cosine similarity between predicted and gold answer embeddings.
                    RAGAS-style metric — measures semantic closeness without needing
                    an LLM judge. Range: 0–1.

Context Relevance : Cosine similarity between question and retrieved context.
                    Measures how well the retriever found relevant context.
                    Same for all 3 models (same retriever). Range: 0–1.

Faithfulness      : Cosine similarity between predicted answer and retrieved context.
                    Measures how grounded the answer is in the retrieved text.
                    High = answer stays close to what was retrieved. Range: 0–1.

Note: RAGAS-style metrics here use sentence-transformer cosine similarity
instead of an LLM judge (LLM-based RAGAS requires OpenAI API key).
""")


METRIC EXPLANATIONS
───────────────────
BLEU              : n-gram overlap between predicted and gold answer.
                    Higher = more word-for-word match.
                    Range: 0–100. Generative models score lower even when correct.

BERTScore F1      : Semantic similarity using multilingual BERT embeddings (lang=kn).
                    More reliable than BLEU for Kannada paraphrased answers.
                    Range: 0–1. > 0.80 is very good.

Answer Similarity : Cosine similarity between predicted and gold answer embeddings.
                    RAGAS-style metric — measures semantic closeness without needing
                    an LLM judge. Range: 0–1.

Context Relevance : Cosine similarity between question and retrieved context.
                    Measures how well the retriever found relevant context.
                    Same for all 3 models (same retriever). Range: 0–1.

Faithfulness      : Cosine similarity between predicted answer and retrieved context.
    

## ── CELL 5 ── Save comparison CSV

In [5]:
csv_path = f'{RESULTS_DIR}/final_comparison.csv'
df.to_csv(csv_path, index=False)
print(f'✅ Comparison table saved → {csv_path}')
print()
print(df.to_string(index=False))

✅ Comparison table saved → /content/drive/MyDrive/AgriQA/rag_comparison_results/final_comparison.csv

                  Model          Type   BLEU  BERTScore F1  Answer Similarity  Context Relevance  Faithfulness
       muril-base-cased extractive_qa 1.0007        0.6047             0.5410               0.83        0.5456
xlm-roberta-base-squad2 extractive_qa 4.8908        0.6826             0.6285               0.83        0.6040
         sarvam-2b-v0.5    generative 2.1260        0.6730             0.7548               0.83        0.7091
